In [1]:
from pathlib import Path
import csv
import shutil

# GitHub hard limit: 100 MiB
SIZE_LIMIT = 100 * 1024 * 1024

# Run notebook from the root of your Git repository
REPO_ROOT = Path.cwd().resolve()

OVERSIZED_DIR = REPO_ROOT / "oversized_files_for_git"
CSV_FILE = OVERSIZED_DIR / "original_locations.csv"

OVERSIZED_DIR.mkdir(exist_ok=True)

# Don't scan .git or our own oversized storage directory
SKIP_DIRS = {".git", "oversized_files_for_git"}

records = []

for path in REPO_ROOT.rglob("*"):
    if not path.is_file():
        continue

    relative = path.relative_to(REPO_ROOT)

    # Skip .git and oversized_files_for_git
    if any(part in SKIP_DIRS for part in relative.parts):
        continue

    size = path.stat().st_size

    if size > SIZE_LIMIT:
        # Preserve the original directory structure inside oversized_files_for_git
        destination = OVERSIZED_DIR / "files" / relative
        destination.parent.mkdir(parents=True, exist_ok=True)

        print(
            f"Moving: {relative} "
            f"({size / 1024**2:.2f} MiB)"
        )

        shutil.move(str(path), str(destination))

        records.append({
            "original_path": str(relative),
            "stored_path": str(destination.relative_to(REPO_ROOT)),
            "size_bytes": size,
            "size_mib": round(size / 1024**2, 2),
        })

# Write manifest
with CSV_FILE.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "original_path",
            "stored_path",
            "size_bytes",
            "size_mib",
        ],
    )
    writer.writeheader()
    writer.writerows(records)

print()
print(f"Moved {len(records)} oversized file(s).")
print(f"Manifest: {CSV_FILE}")
print(f"Storage:  {OVERSIZED_DIR / 'files'}")

Moving: Parcels_view_8092217112315731204.csv (112.65 MiB)
Moving: .nashville_cache\original_parcels\Parcels_view_-889932610375369568.geojson (398.22 MiB)

Moved 2 oversized file(s).
Manifest: C:\Users\micha\Desktop\Nashville_TN_stvrs\oversized_files_for_git\original_locations.csv
Storage:  C:\Users\micha\Desktop\Nashville_TN_stvrs\oversized_files_for_git\files


In [ ]:
from pathlib import Path
import csv
import shutil

REPO_ROOT = Path.cwd().resolve()

OVERSIZED_DIR = REPO_ROOT / "oversized_files_for_git"
CSV_FILE = OVERSIZED_DIR / "original_locations.csv"

if not CSV_FILE.exists():
    raise FileNotFoundError(
        f"Manifest not found: {CSV_FILE}"
    )

restored = 0
missing = 0

with CSV_FILE.open("r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    for row in reader:
        original = REPO_ROOT / row["original_path"]
        stored = REPO_ROOT / row["stored_path"]

        if not stored.exists():
            print(f"MISSING: {stored}")
            missing += 1
            continue

        original.parent.mkdir(parents=True, exist_ok=True)

        # Safety: don't silently overwrite something created since the move
        if original.exists():
            print(f"SKIPPED (already exists): {original}")
            continue

        print(f"Restoring: {row['original_path']}")

        shutil.move(str(stored), str(original))
        restored += 1

print()
print(f"Restored: {restored}")
print(f"Missing:  {missing}")